# **CRISP-DM Case Study: Airline Passenger Satisfaction Prediction**

# 1. Business Understanding

## 1.1 Business Objective

The airline aims to reduce passenger churn and improve customer loyalty by identifying the key factors that influence passenger satisfaction. Instead of investing equally across all service areas, the organization seeks to determine which aspects—such as service quality, seating comfort, Wi-Fi connectivity, and the boarding process—have the greatest impact on overall passenger satisfaction. These insights will support data-driven decision-making and help prioritize improvements that maximize customer experience.

---

## 1.2 Data Mining Goal

The primary objective is to develop a **binary classification model** capable of predicting passenger satisfaction (`Satisfied` or `Neutral/Dissatisfied`) using **23 input features** that describe passenger demographics, flight information, and in-flight service ratings.

In addition to prediction, the project aims to identify and rank the most influential factors affecting passenger satisfaction, enabling the airline to implement targeted operational improvements.

---

## 1.3 Success Criteria

The project will be considered successful if it satisfies the following criteria:

- Achieve a **test accuracy of at least 90%** on unseen data.
- Achieve an **F1-score of at least 0.90**, ensuring a balanced performance between precision and recall.
- Generate an **interpretable ranking of the most influential service factors** affecting passenger satisfaction.
- Validate the robustness of the selected model using **5-fold cross-validation** to ensure consistent performance across multiple data splits.

---

## 1.4 Project Plan

The project will be carried out in the following stages:

### 1. Data Acquisition and Exploration
- Load and examine the Airline Passenger Satisfaction dataset (Maven Analytics version) containing **129,880 passenger records**.
- Perform exploratory data analysis (EDA) to understand feature distributions, relationships, and data quality.

### 2. Data Preparation
- Handle missing values and duplicate records.
- Perform feature engineering where appropriate.
- Encode categorical variables.
- Prepare the dataset for machine learning.

### 3. Model Development
Train and compare the following classification algorithms:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- Naïve Bayes
- Support Vector Machine (SVM)

### 4. Model Evaluation
- Evaluate models using:
  - Accuracy
  - Precision
  - Recall
  - F1-score
  - ROC-AUC
  - Confusion Matrix
- Compare model performance using visualizations.
- Validate the best-performing model using **5-fold cross-validation**.

### 5. Business Evaluation
- Verify whether the selected model meets the defined business success criteria.
- Identify and interpret the most important features influencing passenger satisfaction.

### 6. Segment Analysis
- Analyze model performance across different customer segments, such as:
  - Travel Class
  - Customer Type
  - Type of Travel
- Generate business insights to support strategic decision-making.

### 7. Deployment
- Package the trained model for deployment into an airline satisfaction monitoring system.
- Enable continuous prediction of passenger satisfaction.
- Support data-driven service improvements and operational decision-making.

---

## Expected Deliverables

- Exploratory Data Analysis (EDA)
- Cleaned and preprocessed dataset
- Trained machine learning models
- Model performance comparison
- Cross-validation results
- Feature importance analysis
- Business recommendations
- Deployable prediction model

### **Required Libraries**

In [83]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

In [84]:
import os
import pickle

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

### **Formattings**

In [85]:
# ---------- Global plotting style ----------
sns.set_style("whitegrid")
PALETTE = ["#1F4E79", "#2E86AB", "#5DA9C0", "#A23B72", "#F18F01", "#3B9C6D"]
ACCENT = "#1F4E79"
ACCENT2 = "#C0392B"
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "font.size": 10,
})

### **Functions**

In [86]:
def annotate_bars(ax, fmt="{:.0f}", fontsize=9, color="black", offset=0.01, pct_total=None):
    for p in ax.patches:
        height = p.get_height()
        width = p.get_width()
        # Vertical bars
        if height != 0 and abs(height) >= abs(width):
            x = p.get_x() + width / 2
            y = height
            label = fmt.format(height)
            if pct_total:
                label += f"\n({height / pct_total * 100:.1f}%)"
            ax.annotate(label, (x, y), ha="center", va="bottom",
                        fontsize=fontsize, color=color,
                        xytext=(0, 3), textcoords="offset points")
        else:
            y = p.get_y() + height / 2
            x = width
            label = fmt.format(width)
            ax.annotate(label, (x, y), ha="left", va="center",
                        fontsize=fontsize, color=color,
                        xytext=(3, 0), textcoords="offset points")

def prepare_data(df, label_encoders=None, fit=True):
    df = df.copy()

    if df["Arrival Delay in Minutes"].isnull().any():
        median_delay = df["Arrival Delay in Minutes"].median()
        df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(median_delay)

    service_cols = [
        "Inflight wifi service", "Departure/Arrival time convenient",
        "Ease of Online booking", "Gate location", "Food and drink",
        "Online boarding", "Seat comfort", "Inflight entertainment",
        "On-board service", "Leg room service", "Baggage handling",
        "Checkin service", "Inflight service", "Cleanliness"
    ]
    df["Avg Service Rating"] = df[service_cols].mean(axis=1)
    df["Total Delay"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"]

    cat_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]
    if label_encoders is None:
        label_encoders = {}

    for col in cat_cols:
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            label_encoders[col] = le
        else:
            le = label_encoders[col]
            df[col] = le.transform(df[col])

    if "satisfaction" in df.columns:
        df["satisfaction"] = df["satisfaction"].map({"satisfied": 1, "neutral or dissatisfied": 0})

    return df, label_encoders

## **PHASE 2: DATA UNDERSTANDING**

In [87]:
train_raw = pd.read_csv("data/train.csv")
test_raw = pd.read_csv("data/test.csv")

In [88]:
train_raw.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [89]:
train_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   Gender                             103904 non-null  object 
 3   Customer Type                      103904 non-null  object 
 4   Age                                103904 non-null  int64  
 5   Type of Travel                     103904 non-null  object 
 6   Class                              103904 non-null  object 
 7   Flight Distance                    103904 non-null  int64  
 8   Inflight wifi service              103904 non-null  int64  
 9   Departure/Arrival time convenient  103904 non-null  int64  
 10  Ease of Online booking             103904 non-null  int64  
 11  Gate location                      1039

In [90]:
test_raw.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,19556,Female,Loyal Customer,52,Business travel,Eco,160,5,4,...,5,5,5,5,2,5,5,50,44.0,satisfied
1,1,90035,Female,Loyal Customer,36,Business travel,Business,2863,1,1,...,4,4,4,4,3,4,5,0,0.0,satisfied
2,2,12360,Male,disloyal Customer,20,Business travel,Eco,192,2,0,...,2,4,1,3,2,2,2,0,0.0,neutral or dissatisfied
3,3,77959,Male,Loyal Customer,44,Business travel,Business,3377,0,0,...,1,1,1,1,3,1,4,0,6.0,satisfied
4,4,36875,Female,Loyal Customer,49,Business travel,Eco,1182,2,3,...,2,2,2,2,4,2,4,0,20.0,satisfied


In [91]:
test_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25976 entries, 0 to 25975
Data columns (total 25 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Unnamed: 0                         25976 non-null  int64  
 1   id                                 25976 non-null  int64  
 2   Gender                             25976 non-null  object 
 3   Customer Type                      25976 non-null  object 
 4   Age                                25976 non-null  int64  
 5   Type of Travel                     25976 non-null  object 
 6   Class                              25976 non-null  object 
 7   Flight Distance                    25976 non-null  int64  
 8   Inflight wifi service              25976 non-null  int64  
 9   Departure/Arrival time convenient  25976 non-null  int64  
 10  Ease of Online booking             25976 non-null  int64  
 11  Gate location                      25976 non-null  int

In [92]:
print(f"Train shape: {train_raw.shape}")
print(f"Test shape : {test_raw.shape}")

Train shape: (103904, 25)
Test shape : (25976, 25)


In [93]:
for df in (train_raw, test_raw):
    df.drop(columns=[c for c in ["Unnamed: 0", "id"] if c in df.columns],
            inplace=True, errors="ignore")
    print(f"After dropping columns, shape : {df.shape}")

After dropping columns, shape : (103904, 23)
After dropping columns, shape : (25976, 23)


In [94]:
print("Column dtypes:")
print(train_raw.dtypes.value_counts())

Column dtypes:
int64      17
object      5
float64     1
Name: count, dtype: int64


In [95]:
print("Missing values (train dataset):")
missing = train_raw.isnull().sum()
print(missing[missing > 0])

Missing values (train dataset):
Arrival Delay in Minutes    310
dtype: int64


In [96]:
train_raw["Arrival Delay in Minutes"].fillna(train_raw["Arrival Delay in Minutes"].median(), inplace=True)

In [97]:
print("Missing values (train dataset):")
missing = train_raw.isnull().sum()
print(missing[missing > 0])

Missing values (train dataset):
Series([], dtype: int64)


In [98]:
print("Target class balance (train):")
print(train_raw["satisfaction"].value_counts(normalize=True).round(3))

Target class balance (train):
satisfaction
neutral or dissatisfied    0.567
satisfied                  0.433
Name: proportion, dtype: float64


In [99]:
counts = train_raw["satisfaction"].value_counts()
total = counts.sum()
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(counts.index, counts.values, color=[ACCENT, "#94B8D8"], width=0.55,
              edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, counts.values):
    ax.annotate(f"{val:,}\n({val/total*100:.1f}%)",
                (bar.get_x() + bar.get_width() / 2, val),
                ha="center", va="bottom", fontsize=11, fontweight="bold",
                xytext=(0, 5), textcoords="offset points")
    
ax.set_title("Target Class Distribution (Train Set, n={:,})".format(total))
ax.set_ylabel("Number of Passengers")
ax.set_ylim(0, counts.max() * 1.18)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()
plt.savefig(f"{OUT_DIR}/01_target_distribution.png", dpi=150)
plt.close()

In [100]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, title in zip(
    axes, ["Class", "Type of Travel"],
    ["Satisfaction Rate by Travel Class", "Satisfaction Rate by Type of Travel"]
):
    ct = pd.crosstab(train_raw[col], train_raw["satisfaction"], normalize="index") * 100
    ct = ct[["satisfied", "neutral or dissatisfied"]]
    ct.plot(kind="bar", stacked=True, ax=ax, color=[ACCENT, "#D8D8D8"],
            edgecolor="white", width=0.6, legend=False)
    
    for i, (idx, row) in enumerate(ct.iterrows()):
        cum = 0
        for val in row:
            if val > 4:
                ax.text(i, cum + val / 2, f"{val:.1f}%", ha="center", va="center",
                        fontsize=10, fontweight="bold",
                        color="white" if cum < ct["satisfied"].max() else "black")
            cum += val
    ax.set_title(title)
    ax.set_ylabel("Share of Passengers (%)")
    ax.set_xlabel("")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=0)

handles = [plt.Rectangle((0, 0), 1, 1, color=ACCENT),
           plt.Rectangle((0, 0), 1, 1, color="#D8D8D8")]
fig.legend(handles, ["Satisfied", "Neutral / Dissatisfied"],
           loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.0), fontsize=10, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()
plt.savefig(f"{OUT_DIR}/02_satisfaction_by_class_travel.png", dpi=150)
plt.close()

In [101]:
numeric_cols = train_raw.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12.5, 10))
corr = train_raw[numeric_cols].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f",
            annot_kws={"size": 6.5}, square=True, linewidths=0.3,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
plt.title("Correlation Heatmap of Numeric Features (values = Pearson r)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
plt.savefig(f"{OUT_DIR}/03_correlation_heatmap.png", dpi=150)
plt.close()

In [102]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, col, color in zip(axes, ["Age", "Flight Distance"], [ACCENT, "#3B9C6D"]):
    sns.histplot(train_raw[col], bins=30, kde=True, ax=ax, color=color, alpha=0.75)
    mean_v, med_v = train_raw[col].mean(), train_raw[col].median()
    ax.axvline(mean_v, color=ACCENT2, linestyle="--", linewidth=1.5,
               label=f"Mean = {mean_v:.1f}")
    ax.axvline(med_v, color="black", linestyle=":", linewidth=1.5,
               label=f"Median = {med_v:.1f}")
    ax.set_title(f"{col} Distribution")
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/04_age_distance_distribution.png", dpi=150)
plt.close()

In [103]:
age_bins = [0, 18, 30, 45, 60, 100]
age_labels = ["<18", "18-29", "30-44", "45-59", "60+"]
train_raw["Age Group"] = pd.cut(train_raw["Age"], bins=age_bins, labels=age_labels, right=False)
age_ct = pd.crosstab(train_raw["Age Group"], train_raw["satisfaction"], normalize="index") * 100
age_counts = train_raw["Age Group"].value_counts().reindex(age_labels)

fig, ax = plt.subplots(figsize=(8.5, 5))
bars = ax.bar(age_labels, age_ct.reindex(age_labels)["satisfied"], color=ACCENT,
              edgecolor="white", width=0.6)
for bar, val, n in zip(bars, age_ct.reindex(age_labels)["satisfied"], age_counts):
    ax.annotate(f"{val:.1f}%\n(n={n:,})", (bar.get_x() + bar.get_width() / 2, val),
                ha="center", va="bottom", fontsize=9.5, fontweight="bold",
                xytext=(0, 4), textcoords="offset points")
ax.set_title("Satisfaction Rate by Age Group")
ax.set_ylabel("% Satisfied")
ax.set_xlabel("Age Group")
ax.set_ylim(0, max(age_ct["satisfied"]) * 1.25)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/08_satisfaction_by_age_group.png", dpi=150)
plt.close()

In [104]:
train_raw["Total Delay (raw)"] = (train_raw["Departure Delay in Minutes"] +
                                   train_raw["Arrival Delay in Minutes"].fillna(0))
delay_bins = [-1, 0, 15, 60, 1600]
delay_labels = ["No delay", "1-15 min", "16-60 min", ">60 min"]
train_raw["Delay Bucket"] = pd.cut(train_raw["Total Delay (raw)"], bins=delay_bins, labels=delay_labels)
delay_ct = pd.crosstab(train_raw["Delay Bucket"], train_raw["satisfaction"], normalize="index") * 100
delay_counts = train_raw["Delay Bucket"].value_counts().reindex(delay_labels)

fig, ax = plt.subplots(figsize=(8.5, 5))
bars = ax.bar(delay_labels, delay_ct.reindex(delay_labels)["satisfied"], color="#F18F01",
              edgecolor="white", width=0.6)
for bar, val, n in zip(bars, delay_ct.reindex(delay_labels)["satisfied"], delay_counts):
    ax.annotate(f"{val:.1f}%\n(n={n:,})", (bar.get_x() + bar.get_width() / 2, val),
                ha="center", va="bottom", fontsize=9.5, fontweight="bold",
                xytext=(0, 4), textcoords="offset points")
ax.set_title("Satisfaction Rate by Total Delay (Departure + Arrival)")
ax.set_ylabel("% Satisfied")
ax.set_xlabel("Total Delay Bucket")
ax.set_ylim(0, max(delay_ct["satisfied"]) * 1.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/09_satisfaction_by_delay.png", dpi=150)
plt.close()

In [105]:
train_raw.drop(columns=["Age Group", "Total Delay (raw)", "Delay Bucket"], inplace=True)

## **PHASE 3: DATA PREPARATION**

In [106]:
train_df, encoders = prepare_data(train_raw, fit=True)
test_df, _ = prepare_data(test_raw, label_encoders=encoders, fit=False)

In [107]:
print("Missing values after cleaning (train):", train_df.isnull().sum().sum())
print("New engineered features: 'Avg Service Rating', 'Total Delay'")

Missing values after cleaning (train): 0
New engineered features: 'Avg Service Rating', 'Total Delay'


In [108]:
feature_cols = [c for c in train_df.columns if c != "satisfaction"]
X_train, y_train = train_df[feature_cols], train_df["satisfaction"]
X_test, y_test = test_df[feature_cols], test_df["satisfaction"]

In [109]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)


In [110]:
print(f"\nFinal feature set ({len(feature_cols)} features): {feature_cols}")
print(f"Train set: {X_train.shape}, Test set: {X_test.shape}\n")


Final feature set (24 features): ['Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Avg Service Rating', 'Total Delay']
Train set: (103904, 24), Test set: (25976, 24)



## **PHASE 4: MODELING**

In [111]:
models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=42), False),
    "Decision Tree":       (DecisionTreeClassifier(max_depth=10, random_state=42), False),
    "Random Forest":       (RandomForestClassifier(n_estimators=200, max_depth=15,
                                                     random_state=42, n_jobs=-1), False),
    "Gradient Boosting":   (GradientBoostingClassifier(n_estimators=150, max_depth=3,
                                                          random_state=42), False),
    "Naive Bayes":         (GaussianNB(), False),
    "SVM (linear kernel)": (SVC(kernel="linear", probability=True, random_state=42), True),
}

In [112]:
results = []
fitted_models = {}
SVM_SAMPLE_SIZE = 8000

for name, (model, needs_scaling) in models.items():
    print(f"Training {name} ...")

    if name.startswith("SVM"):
        rng = np.random.RandomState(42)
        idx = rng.choice(len(X_train_scaled), size=SVM_SAMPLE_SIZE, replace=False)
        Xtr, ytr = X_train_scaled.iloc[idx], y_train.iloc[idx]
        Xte = X_test_scaled
    else:
        Xtr, ytr = (X_train_scaled, y_train) if needs_scaling else (X_train, y_train)
        Xte = X_test_scaled if needs_scaling else X_test

    model.fit(Xtr, ytr)
    fitted_models[name] = model

    preds = model.predict(Xte)
    probs = model.predict_proba(Xte)[:, 1] if hasattr(model, "predict_proba") else preds

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs),
    })

Training Logistic Regression ...
Training Decision Tree ...
Training Random Forest ...
Training Gradient Boosting ...
Training Naive Bayes ...
Training SVM (linear kernel) ...


In [113]:
results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False).reset_index(drop=True)
print("\nModel comparison on held-out test set:")
print(results_df.round(4).to_string(index=False))
results_df.to_csv(f"{OUT_DIR}/model_comparison.csv", index=False)


Model comparison on held-out test set:
              Model  Accuracy  Precision  Recall  F1-score  ROC-AUC
      Random Forest    0.9582     0.9646  0.9391    0.9517   0.9929
  Gradient Boosting    0.9495     0.9549  0.9288    0.9417   0.9904
      Decision Tree    0.9453     0.9456  0.9289    0.9372   0.9869
SVM (linear kernel)    0.8726     0.8804  0.8213    0.8498   0.9254
Logistic Regression    0.8605     0.8486  0.8303    0.8394   0.9200
        Naive Bayes    0.8525     0.8453  0.8128    0.8287   0.9140


In [114]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
print(f"\nBest model by F1-score: {best_model_name}\n")


Best model by F1-score: Random Forest



In [115]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
plot_df = results_df.set_index("Model")[metrics_to_plot]

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(plot_df))
width = 0.15
for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, plot_df[metric], width, label=metric,
                   color=PALETTE[i % len(PALETTE)], edgecolor="white")
    for bar, val in zip(bars, plot_df[metric]):
        ax.annotate(f"{val:.3f}", (bar.get_x() + bar.get_width() / 2, val),
                    ha="center", va="bottom", fontsize=7, rotation=90,
                    xytext=(0, 3), textcoords="offset points")
ax.set_xticks(x + width * 2)
ax.set_xticklabels(plot_df.index, rotation=15, ha="right")
ax.set_ylim(0.75, 1.03)
ax.set_ylabel("Score")
ax.set_title("Model Comparison Across All Evaluation Metrics (Held-out Test Set)")
ax.legend(loc="lower right", ncol=5, fontsize=8.5, frameon=True)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/10_model_comparison_bars.png", dpi=150)
plt.close()

In [116]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
top3_names = results_df["Model"].head(3).tolist()
cv_results = {}

for name in top3_names:
    model = type(fitted_models[name])(**fitted_models[name].get_params())
    needs_scale = models[name][1]
    Xcv = X_train_scaled if needs_scale else X_train
    scores = cross_val_score(model, Xcv, y_train, cv=cv, scoring="f1", n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:22s}: F1 per fold = {np.round(scores, 4)} | "
          f"mean = {scores.mean():.4f} | std = {scores.std():.4f}")

Random Forest         : F1 per fold = [0.9537 0.9476 0.9523 0.9501 0.9493] | mean = 0.9506 | std = 0.0022
Gradient Boosting     : F1 per fold = [0.9433 0.9378 0.9411 0.9398 0.9371] | mean = 0.9398 | std = 0.0023
Decision Tree         : F1 per fold = [0.9368 0.9322 0.9354 0.9322 0.9364] | mean = 0.9346 | std = 0.0020


In [117]:
cv_df = pd.DataFrame(cv_results)
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.boxplot(data=cv_df, palette=PALETTE[:3], width=0.45, ax=ax)
sns.stripplot(data=cv_df, color="black", size=6, jitter=True, ax=ax, alpha=0.6)
for i, col in enumerate(cv_df.columns):
    ax.annotate(f"mean={cv_df[col].mean():.4f}", (i, cv_df[col].max()),
                ha="center", va="bottom", fontsize=9, fontweight="bold",
                xytext=(0, 8), textcoords="offset points")
ax.set_title("5-Fold Cross-Validation F1-Score Stability (Top 3 Models)")
ax.set_ylabel("F1-score")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/11_cross_validation.png", dpi=150)
plt.close()

## **PHASE 5: EVALUATION**

In [118]:
needs_scaling = models[best_model_name][1]
Xte_best = X_test_scaled if needs_scaling else X_test
best_preds = best_model.predict(Xte_best)
best_probs = best_model.predict_proba(Xte_best)[:, 1]

print(f"Detailed classification report for best model ({best_model_name}):")
print(classification_report(y_test, best_preds,target_names=["neutral/dissatisfied", "satisfied"]))

Detailed classification report for best model (Random Forest):
                      precision    recall  f1-score   support

neutral/dissatisfied       0.95      0.97      0.96     14573
           satisfied       0.96      0.94      0.95     11403

            accuracy                           0.96     25976
           macro avg       0.96      0.96      0.96     25976
        weighted avg       0.96      0.96      0.96     25976



In [119]:
cm = confusion_matrix(y_test, best_preds)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
labels = np.array([[f"{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)" for j in range(2)] for i in range(2)])

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=labels, fmt="", cmap="Blues",
            xticklabels=["neutral/dissatisfied", "satisfied"],
            yticklabels=["neutral/dissatisfied", "satisfied"],
            annot_kws={"size": 11, "fontweight": "bold"}, cbar_kws={"label": "Count"})
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.title(f"Confusion Matrix - {best_model_name}\n(Accuracy = {accuracy_score(y_test, best_preds):.4f})")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/05_confusion_matrix.png", dpi=150)
plt.close()

In [120]:
plt.figure(figsize=(7, 6))
for i, (name, (model, needs_scale)) in enumerate(models.items()):
    Xte_eval = X_test_scaled if needs_scale else X_test
    if hasattr(fitted_models[name], "predict_proba"):
        probs = fitted_models[name].predict_proba(Xte_eval)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, probs)
        auc = roc_auc_score(y_test, probs)
        lw = 3 if name == best_model_name else 1.5
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})",
                 color=PALETTE[i % len(PALETTE)], linewidth=lw)
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random guess (AUC=0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - All Models (Held-out Test Set)")
plt.legend(fontsize=8.5, loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/06_roc_curves.png", dpi=150)
plt.close()

In [121]:
importance_source = best_model if hasattr(best_model, "feature_importances_") \
    else fitted_models["Random Forest"]
importances = pd.Series(importance_source.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False).head(12)

plt.figure(figsize=(8.5, 6))
bars = plt.barh(importances.index[::-1], importances.values[::-1],
                 color=sns.color_palette("viridis", len(importances)))
for bar, val in zip(bars, importances.values[::-1]):
    plt.annotate(f"{val:.4f}", (val, bar.get_y() + bar.get_height() / 2),
                 ha="left", va="center", fontsize=9, fontweight="bold",
                 xytext=(4, 0), textcoords="offset points")
plt.title(f"Top 12 Feature Importances ({best_model_name})")
plt.xlabel("Importance (Gini-based)")
plt.xlim(0, importances.max() * 1.18)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/07_feature_importance.png", dpi=150)
plt.close()

In [122]:
print("\nTop 10 drivers of passenger satisfaction:")
print(importances.head(10).round(4).to_string())

target_accuracy = 0.90
achieved_accuracy = results_df.iloc[0]["Accuracy"]
meets_criteria = achieved_accuracy >= target_accuracy
print(f"\nSuccess criterion (Accuracy >= {target_accuracy}): "
      f"{'MET' if meets_criteria else 'NOT MET'} (achieved {achieved_accuracy:.4f})\n")


Top 10 drivers of passenger satisfaction:
Online boarding           0.1701
Inflight wifi service     0.1516
Type of Travel            0.1082
Class                     0.0950
Inflight entertainment    0.0609
Avg Service Rating        0.0514
Seat comfort              0.0427
Ease of Online booking    0.0381
Leg room service          0.0371
Customer Type             0.0367

Success criterion (Accuracy >= 0.9): MET (achieved 0.9582)



### **Business segment deep-dive using the trained model**

In [123]:
test_df_eval = test_df.copy()
test_df_eval["pred_satisfied"] = best_preds
test_df_eval["actual_satisfied"] = y_test.values

# Decode Class and Type of Travel back to readable labels for the report
class_decoder = dict(zip(encoders["Class"].transform(encoders["Class"].classes_), encoders["Class"].classes_))
travel_decoder = dict(zip(encoders["Type of Travel"].transform(encoders["Type of Travel"].classes_), encoders["Type of Travel"].classes_))
test_df_eval["Class_label"] = test_df_eval["Class"].map(class_decoder)
test_df_eval["Travel_label"] = test_df_eval["Type of Travel"].map(travel_decoder)

segment_summary = test_df_eval.groupby("Class_label").agg(
    n=("pred_satisfied", "size"),
    predicted_satisfaction_rate=("pred_satisfied", "mean"),
    actual_satisfaction_rate=("actual_satisfied", "mean"),
).round(4)
print(segment_summary.to_string())
segment_summary.to_csv(f"{OUT_DIR}/segment_summary.csv")

                 n  predicted_satisfaction_rate  actual_satisfaction_rate
Class_label                                                              
Business     12495                       0.7028                    0.6952
Eco          11564                       0.1624                    0.1939
Eco Plus      1917                       0.2311                    0.2478


In [124]:
fig, ax = plt.subplots(figsize=(8, 5.5))
x = np.arange(len(segment_summary))
width = 0.32
b1 = ax.bar(x - width/2, segment_summary["actual_satisfaction_rate"] * 100, width,
            label="Actual", color=ACCENT, edgecolor="white")
b2 = ax.bar(x + width/2, segment_summary["predicted_satisfaction_rate"] * 100, width,
            label="Predicted", color="#F18F01", edgecolor="white")
for bars in (b1, b2):
    for bar in bars:
        val = bar.get_height()
        ax.annotate(f"{val:.1f}%", (bar.get_x() + bar.get_width()/2, val),
                    ha="center", va="bottom", fontsize=9, fontweight="bold",
                    xytext=(0, 3), textcoords="offset points")
ax.set_xticks(x)
ax.set_xticklabels(segment_summary.index)
ax.set_ylabel("% Satisfied")
ax.set_title("Actual vs. Model-Predicted Satisfaction Rate by Travel Class\n(Held-out Test Set)")
ax.set_ylim(0, max(segment_summary["actual_satisfaction_rate"].max(),
                    segment_summary["predicted_satisfaction_rate"].max()) * 115)
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/12_predicted_vs_actual_by_class.png", dpi=150)
plt.close()